##### What is CDF ?
- Delta Lake Change Data Feed (CDF)
- Automatically generate CDC feeds about Delta Lake tables.
- Records row-level changes for all data written into a Delta table
    - Row data + metadata (whether row was inserted , deleted or updated)
- select * form table_changes('table_name',start_version, [end_version])
- CDF not enabled by default : 
    - Explicit : CREATE TABLE myTable (id INT, name STRING) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    - Implicit : ALTER TABLE myTable SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    - All New Delta TAbles : spark.databricks.delta.properties.defaults.enableChangeDataFeed
- CDF Retention follows same policy as VACUUM
    - When running VACUUM, CDF data is also deleted.

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
ALTER TABLE customers_silver
SET TBLPROPERTIES (delta.enableChangeDataFeed = TRUE);

In [0]:
%sql
DESCRIBE TABLE EXTENDED customers_silver;

In [0]:
%sql
DESCRIBE HISTORY customers_silver

In [0]:
bookstore.load_new_data()
bookstore.process_bronze()
bookstore.process_orders_silver()
bookstore.process_customers_silver()

SQL way of reading CDF

In [0]:
%sql
select * 
from table_changes("customers_silver",2)

In [0]:
bookstore.load_new_data()
bookstore.process_bronze()
bookstore.process_orders_silver()
bookstore.process_customers_silver()

Python way of reading CDF

In [0]:
cdf_df = (
    spark.readStream
    .format("delta")
    .option("readChangeData", True)
    .option("startingVersion",2)
    .table("customers_silver")
)
display(cdf_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/cdf_{time.time()}")